In [ ]:
# Chunking Tutorial - Complete Guide

This notebook teaches you everything about text chunking for RAG:

## What You'll Learn:
1. What is chunking and why it matters
2. Different chunking strategies (Fixed, Recursive, Semantic)
3. How to choose chunk size and overlap
4. Comparing chunking strategies side-by-side
5. Impact on RAG quality

## Why Chunking Matters:
- Documents are too large for LLM context windows
- Better retrieval = smaller, focused chunks
- Bad chunking = broken context, poor answers
- Good chunking = coherent context, accurate answers

In [ ]:
import sys
sys.path.append('..')

from vectordb.chunking import (
    chunk_text,
    chunk_markdown,
    ChunkingConfig,
    ChunkingStrategy,
    get_chunk_statistics
)
import json

## Part 1: Sample Document

Let's create a sample document to experiment with:

In [ ]:
sample_text = """
# Machine Learning Fundamentals

## Introduction to Machine Learning

Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed. It focuses on the development of computer programs that can access data and use it to learn for themselves.

The process of learning begins with observations or data, such as examples, direct experience, or instruction, in order to look for patterns in data and make better decisions in the future based on the examples that we provide.

## Types of Machine Learning

### Supervised Learning

Supervised learning is the machine learning task of learning a function that maps an input to an output based on example input-output pairs. It infers a function from labeled training data consisting of a set of training examples.

Common algorithms include:
- Linear Regression
- Logistic Regression
- Decision Trees
- Random Forests
- Support Vector Machines
- Neural Networks

### Unsupervised Learning

Unsupervised learning is a type of machine learning that looks for previously undetected patterns in a data set with no pre-existing labels. It works on its own to discover information without any guidance.

Common algorithms include:
- K-Means Clustering
- Hierarchical Clustering
- Principal Component Analysis (PCA)
- Autoencoders

### Reinforcement Learning

Reinforcement learning is an area of machine learning concerned with how intelligent agents ought to take actions in an environment in order to maximize the notion of cumulative reward. It is employed by various software and machines to find the best possible behavior or path it should take in a specific situation.

## Neural Networks

A neural network is a series of algorithms that endeavors to recognize underlying relationships in a set of data through a process that mimics the way the human brain operates. Neural networks can adapt to changing input so the network generates the best possible result without needing to redesign the output criteria.

### Deep Learning

Deep learning is part of a broader family of machine learning methods based on artificial neural networks with representation learning. Learning can be supervised, semi-supervised or unsupervised. Deep learning architectures such as deep neural networks, deep belief networks, recurrent neural networks and convolutional neural networks have been applied to fields including computer vision, speech recognition, natural language processing, and bioinformatics.

## Applications

Machine learning has numerous applications across various industries:

1. Healthcare: Disease diagnosis, drug discovery, personalized medicine
2. Finance: Fraud detection, algorithmic trading, credit scoring
3. E-commerce: Recommendation systems, customer segmentation
4. Transportation: Autonomous vehicles, traffic prediction
5. Natural Language Processing: Chatbots, translation, sentiment analysis
"""

print(f"Document length: {len(sample_text)} characters")
print(f"Document length: {len(sample_text.split())} words")

## Part 2: Understanding Chunking Strategies

### Strategy 1: FIXED
- Splits text into fixed token-sized chunks
- Simple and predictable
- May break sentences/context mid-thought
- Good for: Uniform processing, simple documents

### Strategy 2: RECURSIVE
- Tries to split on natural boundaries (\n\n, \n, sentences, words)
- Preserves context better
- More intelligent splitting
- Good for: Most documents (RECOMMENDED)

### Strategy 3: SEMANTIC (kamradt, cluster_semantic, llm_semantic)
- Splits based on meaning/topic changes
- Best context preservation
- Requires embeddings/LLM (slower, more expensive)
- Good for: Complex documents, highest quality needed

## Part 3: Experiment with FIXED Chunking

In [ ]:
# FIXED chunking with different sizes
print("="*80)
print("FIXED CHUNKING - Small Chunks (256 tokens)")
print("="*80)

config_small = ChunkingConfig(
    strategy=ChunkingStrategy.FIXED,
    chunk_size=256,
    chunk_overlap=50
)

chunks_small = chunk_text(sample_text, "ml_guide.md", config_small)
stats_small = get_chunk_statistics(chunks_small)

print(f"\nTotal chunks: {stats_small['total_chunks']}")
print(f"Avg chunk size: {stats_small['avg_chunk_size']:.0f} characters")
print(f"Min/Max: {stats_small['min_chunk_size']} / {stats_small['max_chunk_size']}")

print("\nFirst 3 chunks:")
for i, chunk in enumerate(chunks_small[:3], 1):
    print(f"\n--- Chunk {i} ---")
    print(chunk.content[:200] + "...")

print("\n" + "="*80)
print("FIXED CHUNKING - Large Chunks (512 tokens)")
print("="*80)

config_large = ChunkingConfig(
    strategy=ChunkingStrategy.FIXED,
    chunk_size=512,
    chunk_overlap=50
)

chunks_large = chunk_text(sample_text, "ml_guide.md", config_large)
stats_large = get_chunk_statistics(chunks_large)

print(f"\nTotal chunks: {stats_large['total_chunks']}")
print(f"Avg chunk size: {stats_large['avg_chunk_size']:.0f} characters")

print("\nFirst chunk:")
print(chunks_large[0].content[:300] + "...")

## Part 4: Experiment with RECURSIVE Chunking

Recursive chunking is smarter - it tries to split on natural boundaries.

In [ ]:
print("="*80)
print("RECURSIVE CHUNKING (RECOMMENDED)")
print("="*80)

config_recursive = ChunkingConfig(
    strategy=ChunkingStrategy.RECURSIVE,
    chunk_size=512,
    chunk_overlap=50,
    separators=["\n## ", "\n### ", "\n\n", "\n", ". ", " ", ""]
)

chunks_recursive = chunk_text(sample_text, "ml_guide.md", config_recursive)
stats_recursive = get_chunk_statistics(chunks_recursive)

print(f"\nTotal chunks: {stats_recursive['total_chunks']}")
print(f"Avg chunk size: {stats_recursive['avg_chunk_size']:.0f} characters")
print(f"Min/Max: {stats_recursive['min_chunk_size']} / {stats_recursive['max_chunk_size']}")

print("\nAll chunks:")
for i, chunk in enumerate(chunks_recursive, 1):
    print(f"\n{'='*80}")
    print(f"Chunk {i} ({len(chunk.content)} chars)")
    print(f"{'='*80}")
    print(chunk.content[:400] + "..." if len(chunk.content) > 400 else chunk.content)

## Part 5: Understanding Chunk Overlap

Overlap helps maintain context between chunks.

In [ ]:
print("="*80)
print("COMPARING OVERLAP: 0 vs 50 vs 100 tokens")
print("="*80)

for overlap in [0, 50, 100]:
    config = ChunkingConfig(
        strategy=ChunkingStrategy.RECURSIVE,
        chunk_size=300,
        chunk_overlap=overlap
    )
    
    chunks = chunk_text(sample_text, "ml_guide.md", config)
    stats = get_chunk_statistics(chunks)
    
    print(f"\nOverlap={overlap} tokens:")
    print(f"  Total chunks: {stats['total_chunks']}")
    print(f"  Avg size: {stats['avg_chunk_size']:.0f} chars")
    
    if len(chunks) >= 2:
        # Show overlap between chunks
        chunk1_end = chunks[0].content[-100:]
        chunk2_start = chunks[1].content[:100]
        print(f"  End of chunk 1: ...{chunk1_end[-50:]}")
        print(f"  Start of chunk 2: {chunk2_start[:50]}...")

## Part 6: Side-by-Side Comparison

Let's compare how different strategies chunk the same text:

In [ ]:
import pandas as pd

# Test different configurations
configs = [
    ("Fixed 256", ChunkingConfig(strategy=ChunkingStrategy.FIXED, chunk_size=256, chunk_overlap=50)),
    ("Fixed 512", ChunkingConfig(strategy=ChunkingStrategy.FIXED, chunk_size=512, chunk_overlap=50)),
    ("Recursive 256", ChunkingConfig(strategy=ChunkingStrategy.RECURSIVE, chunk_size=256, chunk_overlap=50)),
    ("Recursive 512", ChunkingConfig(strategy=ChunkingStrategy.RECURSIVE, chunk_size=512, chunk_overlap=50)),
]

results = []

for name, config in configs:
    chunks = chunk_text(sample_text, "ml_guide.md", config)
    stats = get_chunk_statistics(chunks)
    
    results.append({
        "Strategy": name,
        "Total Chunks": stats['total_chunks'],
        "Avg Size (chars)": f"{stats['avg_chunk_size']:.0f}",
        "Min Size": stats['min_chunk_size'],
        "Max Size": stats['max_chunk_size'],
        "Total Sources": stats['unique_sources']
    })

df = pd.DataFrame(results)
print("\nCHUNKING STRATEGY COMPARISON")
print("="*80)
print(df.to_string(index=False))

print("\n💡 Key Observations:")
print("- Smaller chunks = More chunks = More granular retrieval")
print("- Larger chunks = Fewer chunks = More context per retrieval")
print("- Recursive often creates more uniform chunks")
print("- Fixed is fastest but may break context")

## Part 7: Real File Chunking

Let's chunk an actual markdown file from your project:

In [ ]:
# Chunk your README or any markdown file
from pathlib import Path

# Find a markdown file in your project
readme_path = Path("../README.md")

if readme_path.exists():
    config = ChunkingConfig(
        strategy=ChunkingStrategy.RECURSIVE,
        chunk_size=512,
        chunk_overlap=50
    )
    
    chunks = chunk_markdown(str(readme_path), config)
    stats = get_chunk_statistics(chunks)
    
    print(f"Chunked: {readme_path}")
    print(f"Total chunks: {stats['total_chunks']}")
    print(f"Avg chunk size: {stats['avg_chunk_size']:.0f} characters")
    
    print("\nFirst 3 chunks:")
    for i, chunk in enumerate(chunks[:3], 1):
        print(f"\n--- Chunk {i} ---")
        print(chunk.content[:200] + "...")
else:
    print("README.md not found. Try with your own file path.")

## Part 8: Choosing the Right Strategy

### Recommendations:

| Document Type | Strategy | Chunk Size | Overlap |
|---------------|----------|------------|----------|
| Technical docs | Recursive | 512-1024 | 50-100 |
| Code snippets | Fixed | 256-512 | 20-50 |
| Long articles | Recursive | 1024-2048 | 100-200 |
| Chat logs | Fixed | 128-256 | 10-20 |
| Academic papers | Recursive | 512-1024 | 100 |

### General Rules:

1. **Chunk Size:**
   - Smaller (256-512): Better for precise retrieval, more chunks to search
   - Larger (1024-2048): More context, fewer chunks, may be too broad
   - Sweet spot: **512 tokens** for most documents

2. **Overlap:**
   - No overlap: Risk losing context at boundaries
   - Too much overlap: Redundancy, more storage/compute
   - Sweet spot: **10-20% of chunk size**

3. **Strategy:**
   - **Start with RECURSIVE** - best balance
   - Use FIXED for simple/uniform documents
   - Use SEMANTIC only if you need perfect context (costs more)

## Part 9: Impact on RAG Quality

Let's see how chunking affects what gets retrieved:

In [ ]:
# Simulate a search query
query = "supervised learning algorithms"

print(f"Query: '{query}'\n")
print("="*80)

# Test with different chunk sizes
for chunk_size in [256, 512, 1024]:
    config = ChunkingConfig(
        strategy=ChunkingStrategy.RECURSIVE,
        chunk_size=chunk_size,
        chunk_overlap=50
    )
    
    chunks = chunk_text(sample_text, "ml_guide.md", config)
    
    # Find chunks containing query terms
    relevant_chunks = [
        c for c in chunks 
        if any(term.lower() in c.content.lower() for term in query.split())
    ]
    
    print(f"\nChunk size: {chunk_size} tokens")
    print(f"Relevant chunks found: {len(relevant_chunks)}")
    
    if relevant_chunks:
        print(f"\nBest matching chunk ({len(relevant_chunks[0].content)} chars):")
        print(relevant_chunks[0].content[:300] + "...")
        print("\n" + "="*80)

print("\n💡 Notice:")
print("- Smaller chunks: More focused but may lack context")
print("- Larger chunks: More context but may include irrelevant info")
print("- The 'right' size depends on your use case!")

## Part 10: Your Turn - Experiment!

Try chunking with your own text:

In [ ]:
# YOUR EXPERIMENT HERE

your_text = """
Paste your own text here...
"""

# Try different configurations
my_config = ChunkingConfig(
    strategy=ChunkingStrategy.RECURSIVE,  # Try FIXED or RECURSIVE
    chunk_size=512,                       # Adjust size
    chunk_overlap=50                      # Adjust overlap
)

my_chunks = chunk_text(your_text, "my_doc.md", my_config)

print(f"Total chunks: {len(my_chunks)}")
for i, chunk in enumerate(my_chunks, 1):
    print(f"\n--- Chunk {i} ---")
    print(chunk.content[:200])

## Summary

### What You Learned:

✅ **Chunking strategies**: Fixed, Recursive, Semantic  
✅ **Chunk size impact**: Smaller = precise, Larger = contextual  
✅ **Overlap importance**: Maintains context between chunks  
✅ **How to compare**: Statistics and visualization  
✅ **Best practices**: Start with Recursive, 512 tokens, 50 overlap  

### Next Steps:

1. **Next Notebook**: `2_embeddings_tutorial.ipynb` - Convert chunks to vectors
2. **Experiment**: Try chunking your own documents
3. **Optimize**: Test different configs for your specific use case

### Quick Reference:

```python
from vectordb.chunking import chunk_markdown, ChunkingConfig, ChunkingStrategy

# Recommended for most use cases
config = ChunkingConfig(
    strategy=ChunkingStrategy.RECURSIVE,
    chunk_size=512,
    chunk_overlap=50
)

chunks = chunk_markdown("path/to/file.md", config)
```